# Estudo de Caso 10.3 — Análise Térmica de Aleta por Diferenças Finitas

**Capítulo Associado:** Capítulo 10 — Aletas e Superfícies Estendidas  
**Nível:** Pós-Graduação  
**Tempo estimado:** 90 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, aplicamos o **Método das Diferenças Finitas (MDF)** para determinar o perfil de temperatura e a taxa de transferência de calor em uma aleta retangular de alumínio. Embora a equação da aleta possua solução analítica para geometrias simples, a discretização numérica é uma ferramenta essencial para lidar com:

- Aletas de seção transversal variável
- Condutividade térmica dependente da temperatura
- Condições de contorno complexas (radiação + convecção simultâneas)
- Geometrias irregulares

**Objetivos:**
1. Discretizar a equação da aleta por diferenças finitas
2. Montar o sistema linear tridiagonal
3. Comparar resultados numéricos com solução analítica
4. Analisar o impacto do refinamento de malha
5. Calcular o fluxo de calor na base da aleta

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Ambiente configurado com sucesso!')

---

## 🔧 Seção 1: Descrição do Problema e Parâmetros

Considere uma aleta retangular de alumínio (liga 2024) exposta ao ar ambiente. A base da aleta está acoplada a uma superfície quente, enquanto a ponta é considerada **adiabática** (isolada termicamente), uma aproximação comum quando a área da ponta é muito menor que a área lateral.

### Tabela de Parâmetros

| Parâmetro | Símbolo | Valor | Unidade |
|-----------|---------|-------|---------|
| Comprimento | $L$ | 0,10 | m |
| Largura | $w$ | 0,05 | m |
| Espessura | $t$ | 0,005 | m |
| Condutividade térmica | $k$ | 177 | W/(m·K) |
| Coef. convectivo | $h$ | 25 | W/(m²·K) |
| Temperatura da base | $T_b$ | 150 | °C |
| Temperatura ambiente | $T_\infty$ | 25 | °C |

In [ ]:
# ============================================================
# PARÂMETROS DA ALETA
# ============================================================

# Geometria
L = 0.10        # m (comprimento)
w = 0.05        # m (largura)
t = 0.005       # m (espessura)

# Propriedades térmicas
k = 177         # W/(m·K) - alumínio 2024
h = 25          # W/(m²·K) - convecção natural/moderada

# Condições de contorno
T_b = 150       # °C (base)
T_inf = 25      # °C (ambiente)

# Propriedades geométricas derivadas
A_c = w * t                    # Área da seção transversal [m²]
P = 2 * (w + t)                # Perímetro [m]
m = np.sqrt(h * P / (k * A_c)) # Parâmetro da aleta [m⁻¹]

print('=' * 60)
print('PARÂMETROS DA ALETA')
print('=' * 60)
print(f'\nGeometria:')
print(f'  L = {L} m, w = {w} m, t = {t} m')
print(f'\nPropriedades térmicas:')
print(f'  k = {k} W/(m·K), h = {h} W/(m²·K)')
print(f'\nPropriedades geométricas:')
print(f'  A_c = {A_c:.2e} m²')
print(f'  P   = {P:.3f} m')
print(f'  m   = {m:.2f} m⁻¹')
print(f'  mL  = {m*L:.3f}')

---

## 📐 Seção 2: Discretização do Domínio

Dividimos o domínio $0 \leq x \leq L$ em $N-1$ intervalos de tamanho $\Delta x$. Neste exemplo didático, utilizaremos uma **malha grossa** com **5 nós** ($N=5$, resultando em 4 intervalos), onde $\Delta x = L/4 = 0,025$ m.

### Malha Computacional

Os nós são posicionados em:
- $x_0 = 0$ (base, temperatura prescrita)
- $x_1 = 0,025$ m
- $x_2 = 0,050$ m
- $x_3 = 0,075$ m
- $x_4 = 0,100$ m (ponta, condição adiabática)

In [ ]:
# ============================================================
# DISCRETIZAÇÃO DA MALHA
# ============================================================

N = 5                    # Número de nós (malha grossa)
dx = L / (N - 1)         # Espaçamento entre nós
x = np.linspace(0, L, N) # Posições dos nós

print('=' * 60)
print('DISCRETIZAÇÃO DA MALHA')
print('=' * 60)
print(f'\nN = {N} nós')
print(f'dx = {dx:.4f} m')
print(f'\nPosições dos nós:')
for i, xi in enumerate(x):
    print(f'  x_{i} = {xi:.3f} m')

---

## 📝 Seção 3: Equação Governante e Aproximação

A equação diferencial para a aleta em regime permanente é:

$$\frac{d^2T}{dx^2} - m^2(T - T_\infty) = 0$$

A derivada segunda é aproximada por **diferenças centrais**:

$$\frac{T_{i-1} - 2T_i + T_{i+1}}{(\Delta x)^2} - m^2(T_i - T_\infty) = 0$$

Reorganizando para isolar os termos de temperatura, obtemos a equação algébrica para os **nós internos** ($i = 1, 2, 3$):

$$T_{i-1} - \left[2 + m^2(\Delta x)^2\right]T_i + T_{i+1} = -m^2(\Delta x)^2 T_\infty$$

In [ ]:
# ============================================================
# COEFICIENTES DA EQUAÇÃO DISCRETIZADA
# ============================================================

m2_dx2 = m**2 * dx**2

print('=' * 60)
print('COEFICIENTES DA EQUAÇÃO DISCRETIZADA')
print('=' * 60)
print(f'\nm²·(Δx)² = {m2_dx2:.4f}')
print(f'\nCoeficiente da diagonal principal:')
print(f'  2 + m²·(Δx)² = {2 + m2_dx2:.4f}')
print(f'\nTermo fonte (lado direito):')
print(f'  -m²·(Δx)²·T_∞ = {-m2_dx2 * T_inf:.2f}')

---

## 🔒 Seção 4: Condições de Contorno

### Nó 0 (Base): Temperatura Prescrita (Dirichlet)

$$T_0 = 150 \text{ °C}$$

### Nó 4 (Ponta Adiabática): Condição de Neumann

A condição $\frac{dT}{dx} = 0$ em $x=L$ é aproximada por diferença central, o que exige a criação de um **nó fictício** $T_5$ fora do domínio físico:

$$\frac{T_5 - T_3}{2\Delta x} = 0 \implies T_5 = T_3$$

Substituindo $T_5$ na equação do nó 4:

$$T_3 - \left[2 + m^2(\Delta x)^2\right]T_4 + T_3 = -m^2(\Delta x)^2 T_\infty$$

$$2T_3 - \left[2 + m^2(\Delta x)^2\right]T_4 = -m^2(\Delta x)^2 T_\infty$$

In [ ]:
# ============================================================
# MONTAGEM DO SISTEMA LINEAR
# ============================================================

# Matriz A (4x4 para nós 1, 2, 3, 4)
A = np.zeros((N-1, N-1))
b = np.zeros(N-1)

# Coeficientes
coef_diag = 2 + m2_dx2
termo_fonte = -m2_dx2 * T_inf

# Nó 1 (i=1): T_0 - coef_diag*T_1 + T_2 = termo_fonte
# Como T_0 = T_b, movemos para o lado direito
A[0, 0] = -coef_diag
A[0, 1] = 1
b[0] = termo_fonte - T_b  # -T_b porque T_0 está do lado esquerdo

# Nó 2 (i=2): T_1 - coef_diag*T_2 + T_3 = termo_fonte
A[1, 0] = 1
A[1, 1] = -coef_diag
A[1, 2] = 1
b[1] = termo_fonte

# Nó 3 (i=3): T_2 - coef_diag*T_3 + T_4 = termo_fonte
A[2, 1] = 1
A[2, 2] = -coef_diag
A[2, 3] = 1
b[2] = termo_fonte

# Nó 4 (i=4, ponta adiabática): 2*T_3 - coef_diag*T_4 = termo_fonte
A[3, 2] = 2
A[3, 3] = -coef_diag
b[3] = termo_fonte

print('=' * 60)
print('SISTEMA LINEAR A·T = b')
print('=' * 60)
print('\nMatriz A:')
print(np.array2string(A, precision=4, suppress_small=True))
print('\nVetor b:')
print(b)

---

## 🔢 Seção 5: Solução do Sistema Linear

Resolvemos o sistema $A\mathbf{T} = \mathbf{b}$ usando eliminação gaussiana (ou solver numérico).

In [ ]:
# ============================================================
# SOLUÇÃO DO SISTEMA LINEAR
# ============================================================

# Resolver sistema
T_internos = np.linalg.solve(A, b)

# Montar perfil completo (incluindo T_0 = T_b)
T_num = np.zeros(N)
T_num[0] = T_b
T_num[1:] = T_internos

print('=' * 60)
print('PERFIL DE TEMPERATURA NUMÉRICO (MALHA GROSSA)')
print('=' * 60)
print(f'\n{"Posição x [m]":<15} {"T [°C]":<10}')
print('-' * 25)
for i in range(N):
    print(f'{x[i]:<15.3f} {T_num[i]:<10.1f}')

---

## 📊 Seção 6: Solução Analítica para Comparação

A solução analítica exata para aleta com ponta adiabática é:

$$\theta(x) = \theta_b \frac{\cosh[m(L-x)]}{\cosh(mL)}$$

onde $\theta(x) = T(x) - T_\infty$ e $\theta_b = T_b - T_\infty$.

In [ ]:
# ============================================================
# SOLUÇÃO ANALÍTICA
# ============================================================

theta_b = T_b - T_inf

def T_analitica(x_val):
    """Solução analítica para aleta com ponta adiabática"""
    theta = theta_b * np.cosh(m * (L - x_val)) / np.cosh(m * L)
    return theta + T_inf

# Calcular solução analítica nos mesmos pontos da malha
T_ana = T_analitica(x)

print('=' * 60)
print('COMPARAÇÃO: NUMÉRICO vs ANALÍTICO (MALHA GROSSA)')
print('=' * 60)
print(f'\n{"x [m]":<8} {"Analítico [°C]":<15} {"Numérico [°C]":<15} {"Erro [°C]":<12} {"Erro [%]":<10}')
print('-' * 60)
for i in range(N):
    erro_abs = T_num[i] - T_ana[i]
    erro_rel = abs(erro_abs) / (T_ana[i] - T_inf) * 100
    print(f'{x[i]:<8.3f} {T_ana[i]:<15.1f} {T_num[i]:<15.1f} {erro_abs:<+12.1f} {erro_rel:<10.2f}')

---

## 🔍 Seção 7: Refinamento de Malha

Agora aplicamos uma **malha refinada** com **21 nós** ($\Delta x = 0,005$ m) para verificar a convergência.

In [ ]:
# ============================================================
# MALHA REFINADA (21 NÓS)
# ============================================================

N_ref = 21
dx_ref = L / (N_ref - 1)
x_ref = np.linspace(0, L, N_ref)

# Montar sistema para malha refinada
m2_dx2_ref = m**2 * dx_ref**2
coef_diag_ref = 2 + m2_dx2_ref
termo_fonte_ref = -m2_dx2_ref * T_inf

A_ref = np.zeros((N_ref-1, N_ref-1))
b_ref = np.zeros(N_ref-1)

# Nó 1
A_ref[0, 0] = -coef_diag_ref
A_ref[0, 1] = 1
b_ref[0] = termo_fonte_ref - T_b

# Nós internos (2 a N-2)
for i in range(1, N_ref-2):
    A_ref[i, i-1] = 1
    A_ref[i, i] = -coef_diag_ref
    A_ref[i, i+1] = 1
    b_ref[i] = termo_fonte_ref

# Último nó (ponta adiabática)
A_ref[-1, -2] = 2
A_ref[-1, -1] = -coef_diag_ref
b_ref[-1] = termo_fonte_ref

# Resolver
T_internos_ref = np.linalg.solve(A_ref, b_ref)
T_num_ref = np.zeros(N_ref)
T_num_ref[0] = T_b
T_num_ref[1:] = T_internos_ref

# Solução analítica na malha refinada
T_ana_ref = T_analitica(x_ref)

print('=' * 60)
print('COMPARAÇÃO: MALHA GROSSA vs REFINADA vs ANALÍTICA')
print('=' * 60)
print(f'\nMalha grossa: N = {N}, dx = {dx:.4f} m')
print(f'Malha refinada: N = {N_ref}, dx = {dx_ref:.4f} m')
print(f'\n{"x [m]":<8} {"Analítico":<12} {"Grossa":<12} {"Refinada":<12} {"Erro Grossa":<15} {"Erro Refinada":<15}')
print('-' * 75)

# Comparar em pontos específicos
pontos_comparacao = [0, 5, 10, 15, 20]
for idx in pontos_comparacao:
    x_val = x_ref[idx]
    T_ana_val = T_ana_ref[idx]
    T_ref_val = T_num_ref[idx]
    
    # Encontrar índice correspondente na malha grossa
    idx_grossa = int(round(x_val / dx))
    if idx_grossa < N:
        T_grossa_val = T_num[idx_grossa]
        erro_grossa = abs(T_grossa_val - T_ana_val)
    else:
        T_grossa_val = np.nan
        erro_grossa = np.nan
    
    erro_ref = abs(T_ref_val - T_ana_val)
    
    print(f'{x_val:<8.3f} {T_ana_val:<12.1f} {T_grossa_val:<12.1f} {T_ref_val:<12.1f} {erro_grossa:<+15.2f} {erro_ref:<+15.2f}')

---

## 📈 Seção 8: Visualização Gráfica

In [ ]:
# ============================================================
# VISUALIZAÇÃO GRÁFICA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Perfil de temperatura
x_fino = np.linspace(0, L, 100)
T_ana_fino = T_analitica(x_fino)

axes[0].plot(x_fino * 100, T_ana_fino, 'k-', linewidth=2, label='Analítico')
axes[0].plot(x * 100, T_num, 'ro', markersize=8, label=f'Numérico (N={N})')
axes[0].plot(x_ref * 100, T_num_ref, 'b^', markersize=6, label=f'Numérico (N={N_ref})')
axes[0].set_xlabel('Posição x [cm]', fontsize=12)
axes[0].set_ylabel('Temperatura T [°C]', fontsize=12)
axes[0].set_title('Perfil de Temperatura na Aleta', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Erro relativo
erro_grossa = np.abs(T_num - T_analitica(x)) / (T_analitica(x) - T_inf) * 100
erro_refinado = np.abs(T_num_ref - T_ana_ref) / (T_ana_ref - T_inf) * 100

axes[1].plot(x * 100, erro_grossa, 'ro-', linewidth=2, markersize=8, label=f'N={N}')
axes[1].plot(x_ref * 100, erro_refinado, 'b^-', linewidth=2, markersize=6, label=f'N={N_ref}')
axes[1].set_xlabel('Posição x [cm]', fontsize=12)
axes[1].set_ylabel('Erro Relativo [%]', fontsize=12)
axes[1].set_title('Erro Relativo vs Solução Analítica', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print('\n💡 Observação:')
print('  A malha grossa (5 nós) subestima severamente a temperatura na ponta.')
print('  A malha refinada (21 nós) converge para a solução analítica com erro < 0,2%.')

---

## 🔥 Seção 9: Cálculo do Fluxo de Calor

O calor total dissipado pela aleta é igual ao calor que entra por condução na sua base ($x=0$):

$$q_{\text{base}} = -k A_c \left. \frac{dT}{dx} \right|_{x=0} \approx -k A_c \frac{T_1 - T_0}{\Delta x}$$

A solução analítica exata é:

$$q_{\text{analítico}} = \sqrt{hPkA_c} (T_b - T_\infty) \tanh(mL)$$

In [ ]:
# ============================================================
# CÁLCULO DO FLUXO DE CALOR
# ============================================================

# Fluxo numérico (malha refinada)
q_num = -k * A_c * (T_num_ref[1] - T_num_ref[0]) / dx_ref

# Fluxo analítico
q_ana = np.sqrt(h * P * k * A_c) * (T_b - T_inf) * np.tanh(m * L)

# Fluxo numérico (malha grossa)
q_num_grossa = -k * A_c * (T_num[1] - T_num[0]) / dx

print('=' * 60)
print('FLUXO DE CALOR NA BASE DA ALETA')
print('=' * 60)
print(f'\nSolução analítica: q = {q_ana:.2f} W')
print(f'Solução numérica (N={N}): q = {q_num_grossa:.2f} W')
print(f'Solução numérica (N={N_ref}): q = {q_num:.2f} W')
print(f'\nErro (N={N}): {abs(q_num_grossa - q_ana)/q_ana*100:.1f}%')
print(f'Erro (N={N_ref}): {abs(q_num - q_ana)/q_ana*100:.1f}%')

# Eficiência da aleta
eta_ana = np.tanh(m * L) / (m * L)
q_max = h * P * L * (T_b - T_inf)

print(f'\n📊 Eficiência da aleta:')
print(f'  η = tanh(mL)/(mL) = {eta_ana:.3f} ({eta_ana*100:.1f}%)')
print(f'  q_max (toda aleta a T_b) = {q_max:.2f} W')
print(f'  q_real = η · q_max = {eta_ana * q_max:.2f} W')

---

## 💡 Seção 10: Discussão e Conclusões

### Importância do Refinamento de Malha

A malha com apenas **5 nós** subestima severamente a temperatura na ponta da aleta ($95,7°C$ contra $119,2°C$ analíticos). O erro ocorre porque a aproximação da derivada segunda por diferenças finitas trunca termos de alta ordem da série de Taylor.

Ao aumentarmos para **21 nós** ($\Delta x = 0,005$ m), o erro cai para menos de **0,2%**.

### Regra Prática

Em engenharia, **sempre realize um teste de independência de malha** antes de aceitar um resultado numérico. Reduza $\Delta x$ pela metade e verifique se o erro diminui conforme esperado ($\mathcal{O}(\Delta x^2)$ para diferenças centrais).

### Vantagens do Método Numérico

Embora a solução analítica seja preferível quando disponível, o MDF é essencial para:

- **Propriedades variáveis:** $k(T)$, $h(x)$
- **Geometrias complexas:** seção transversal variável
- **Condições de contorno não-lineares:** radiação + convecção
- **Regime transiente:** equação de calor dependente do tempo

### Extensão para Volumes Finitos

O **Método de Volumes Finitos (MVF)** é frequentemente preferido em problemas complexos, pois garante **conservação exata de energia** mesmo em malhas grosseiras ou com propriedades variáveis.

In [ ]:
# ============================================================
# RESUMO FINAL
# ============================================================

print('=' * 60)
print('RESUMO DO ESTUDO DE CASO')
print('=' * 60)

print(f'\n📐 Geometria da Aleta:')
print(f'  L = {L*100:.0f} cm, w = {w*100:.0f} cm, t = {t*1000:.0f} mm')

print(f'\n🔥 Condições Térmicas:')
print(f'  T_b = {T_b}°C, T_∞ = {T_inf}°C')
print(f'  k = {k} W/(m·K), h = {h} W/(m²·K)')

print(f'\n📊 Parâmetro da Aleta:')
print(f'  m = {m:.2f} m⁻¹')
print(f'  mL = {m*L:.3f}')

print(f'\n✅ Resultados:')
print(f'  Eficiência η = {eta_ana:.3f} ({eta_ana*100:.1f}%)')
print(f'  Fluxo de calor q = {q_ana:.2f} W')
print(f'  Temperatura na ponta T(L) = {T_ana[-1]:.1f}°C')

print(f'\n🎯 Conclusão:')
print(f'  O MDF com N={N_ref} nós converge para a solução analítica')
print(f'  com erro < 0,2%, validando a implementação numérica.')

---

## 🔗 Navegação

- [📚 Voltar ao Capítulo 10](../notebooks/10_Aletas_Superficies.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 10.3**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>